In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np

import json
from shapely.geometry import shape

import skmob
from skmob.utils import utils, constants
from skmob.models.gravity import Gravity

In [ ]:
# Укажите путь к вашему файлу
file_path = 'response1.json'

# Загрузите JSON из файла
with open(file_path, 'r', encoding='utf-8') as file:
    json_data = json.load(file)

# Преобразуйте строку в словарь (если JSON внутри строки)
feature_collection1 = json.loads(json_data[0])
feature_collection2 = json.loads(json_data[1])
feature_collection3 = json.loads(json_data[2])

In [ ]:
gdf1 = gpd.GeoDataFrame.from_features(feature_collection1['features'], crs='EPSG:4326')
gdf2 = gpd.GeoDataFrame.from_features(feature_collection1['features'], crs='EPSG:4326')
gdf3 = gpd.GeoDataFrame.from_features(feature_collection3['features'], crs='EPSG:4326')

In [ ]:
pd.options.display.max_columns = None
gdf1.iloc[:, 20:].describe()

In [ ]:
gdf1['Исходящая'] = gdf1['Исходящая. Моложе трудоспособного возраста'] + \
                    gdf1['Исходящая. Трудоспособного возраста'] + \
                    gdf1['Исходящая. Старше трудоспособного возраста']

gdf1['Входящая'] = gdf1['Входящая. Моложе трудоспособного возраста'] + \
                    gdf1['Входящая. Трудоспособного возраста'] + \
                    gdf1['Входящая. Старше трудоспособного возраста']

# gdf1.explore('Исходящая')

In [ ]:
gdf1[['Исходящая', 'Входящая']].sum()

In [ ]:
name_mapping = dict(zip(gdf2['territory_id'], gdf2['name']))
gdf3['From Название территории'] = gdf3['from_territory_id'].map(name_mapping)
gdf3['To Название территории'] = gdf3['to_territory_id'].map(name_mapping)
gdf3.dropna(axis=0, inplace=True)
# gdf3['n_people'] = gdf3['n_people'] / gdf3['n_people'].max()
# print(gdf3[['from_territory_id', 'to_territory_id', 'From Название территории', 'To Название территории']])

In [ ]:
gdf3 = gdf3[ 
    (gdf3['from_territory_id'] != 0)\
    & (gdf3['to_territory_id'] != 0)\
    # & (gdf3['from_territory_id'] != 3138)\
    # & (gdf3['to_territory_id'] != 3138)
    ]

In [ ]:
import folium
from folium.plugins import HeatMap
import geopandas as gpd
from shapely.geometry import LineString
import branca.colormap as cm

# Создаем базовую карту
m = folium.Map(location=[59.9343, 30.3351], zoom_start=7)  # Центр на Санкт-Петербург

# Создаем градиент цветов для n_people
colormap = cm.linear.YlOrRd_09.scale(
    gdf3['n_people'].min(),  # Минимальное значение n_people
    gdf3['n_people'].max()   # Максимальное значение n_people
)

# Добавляем линии на карту
for idx, row in gdf3.iterrows():
    # Получаем координаты линии
    line = row['geometry']
    if line.geom_type == 'LineString':
        coords = list(line.coords)
        # Преобразуем координаты в формат [lat, lon]
        coords = [[y, x] for x, y in coords]  # Shapely использует (x, y), а folium — (lat, lon)

        # Создаем линию с градиентом цвета и толщиной, зависящей от n_people
        folium.PolyLine(
            locations=coords,
            color=colormap(row['n_people']),  # Цвет линии
            weight=row['n_people'] / 200,       # Толщина линии (масштабируем для визуализации)
            opacity=0.8,
            tooltip=f"From: {row['From Название территории']}, To: {row['To Название территории']}, People: {row['n_people']}"
        ).add_to(m)

# Добавляем легенду (градиент цветов)
colormap.caption = 'Количество перемещений (n_people)'
m.add_child(colormap)

m

### Gravity

In [ ]:
gravity_singly = Gravity(gravity_type='singly constrained', deterrence_func_args=[-1.994715203191313], origin_exp=1.0, destination_exp=0.6471759552223115)
print(gravity_singly)

In [ ]:
gdf1['Лечебно-профилактические организации (шт.)'].describe()

In [ ]:
np.random.seed(0)

synth_fdf = gravity_singly.generate(gdf1,
                                   tile_id_column='name',
                                   tot_outflows_column='Исходящая',
                                   relevance_column= 'Лечебно-профилактические организации (шт.)',
                                   out_format='flows')

synth_fdf['flow'] = round(synth_fdf['flow'])
synth_fdf = synth_fdf[synth_fdf['flow'] > 0]
synth_fdf[synth_fdf['origin'].isin(['Муринское городское поселение'])].plot_flows(tiles='CartoDB dark_matter', flow_weight=0.5)

In [ ]:
# Фильтруем строки, где name == 'Муринское городское поселение'
filtered_df = synth_fdf[synth_fdf['origin'] == 'Муринское городское поселение']
filtered_df.sort_values(['flow', 'destination'])

In [ ]:
gdf1.loc[gdf1['name'] == 'Муринское городское поселение', 'Лечебно-профилактические организации (шт.)'] = 1000
gdf1[gdf1['name'] == 'Муринское городское поселение']

In [ ]:
np.random.seed(0)

synth_fdf2 = gravity_singly.generate(gdf1,
                                   tile_id_column='name',
                                   tot_outflows_column='Исходящая',
                                   relevance_column= 'Число спортивных сооружений (шт.)',
                                   out_format='flows')

synth_fdf2['flow'] = round(synth_fdf2['flow'])
synth_fdf2 = synth_fdf2[synth_fdf2['flow'] > 0]
synth_fdf2[synth_fdf2['origin'].isin(['Муринское городское поселение'])].plot_flows(tiles='CartoDB dark_matter', flow_weight=0.5)

In [ ]:
# Фильтруем строки, где name == 'Муринское городское поселение'
filtered_df2 = synth_fdf2[synth_fdf2['origin'] == 'Муринское городское поселение']
filtered_df2.sort_values(['flow', 'destination'])

In [ ]:
# Создаем множество кортежей для быстрого поиска
common_keys = set(zip(synth_fdf['origin'], synth_fdf['destination']))

# Фильтруем synth_fdf2, оставляя только строки, где комбинация origin и destination есть в synth_fdf
filtered_synth_fdf = synth_fdf2[
    synth_fdf2.apply(lambda row: (row['origin'], row['destination']) in common_keys, axis=1)
].copy()  # Используем .copy(), чтобы избежать предупреждений

# Создаем словарь для быстрого доступа к значениям flow в synth_fdf
flow_dict = {(row['origin'], row['destination']): row['flow'] for _, row in synth_fdf.iterrows()}

# Вычитаем значения flow
filtered_synth_fdf['flow'] = filtered_synth_fdf.apply(
    lambda row: row['flow'] - flow_dict.get((row['origin'], row['destination']), float('nan')),
    axis=1
)

# Удаляем строки с NaN (если такие есть)
# filtered_synth_fdf.dropna(subset=['flow'], inplace=True)

# Фильтруем по 'Муринское городское поселение' и строим график
filtered_synth_fdf[filtered_synth_fdf['origin'] == 'Муринское городское поселение'].plot_flows(tiles='CartoDB dark_matter', flow_weight=0.5)

In [ ]:
# Создаем множество кортежей для быстрого поиска
common_keys = set(zip(synth_fdf['origin'], synth_fdf['destination']))

# Фильтруем synth_fdf2, оставляя только строки, где комбинация origin и destination есть в synth_fdf
filtered_synth_fdf = synth_fdf2[
    synth_fdf2.apply(lambda row: (row['origin'], row['destination']) in common_keys, axis=1)
]

# Результат
filtered_synth_fdf['flow'] = filtered_synth_fdf['flow'] - synth_fdf['flow']
filtered_synth_fdf.dropna(inplace=True)
filtered_synth_fdf[filtered_synth_fdf['origin'].isin(['Муринское городское поселение'])].plot_flows(tiles='CartoDB dark_matter', flow_weight=0.5)

In [ ]:
res = filtered_df['flow'] - filtered_df2['flow']
res.describe()

In [ ]:
synth_fdf.tessellation = synth_fdf.tessellation.explode()

In [ ]:
print(synth_fdf.shape)
synth_fdf.plot_flows(tiles='CartoDB dark_matter', min_flow=50, flow_weight=0.5)

In [ ]:
pd.options.display.max_columns = None
gdf1.drop(columns=['geometry', 'centre_point', 'name'])

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Создаём копию исходного DataFrame
df = gdf1.copy()

# 1. Рассчитываем нормированные показатели на 1000 человек
population = df['Численность населения (чел.)']

# Социально-культурные фичи

df['Рестораны'] = df['Количество мест в ресторанах кафе барах (место)'] / population
df['Спорт'] = df['Число спортивных сооружений (шт.)'] / population
df['Быт'] = df['Объекты бытового обслуживания (шт.)'] / population
df['Инвестиции'] = df['Инвестиции в основной капитал (тыс. руб.)'] / population
df['Медицина'] = df['Лечебно-профилактические организации (шт.)'] / population

# Замена бесконечных значений (если population=0)
df.replace([np.inf, -np.inf], 0, inplace=True)

# 2. Нормализация признаков
features = [
    'Рестораны',
    'Спорт',
    'Быт',
    'Инвестиции',
    'Медицина',
]

scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(df[features])

# 3. Весовые коэффициенты 
beta = np.array([
    0.20,  # Рестораны
    0.20,  # Спорт
    0.20,  # Быт
    0.20,  # Инвестиции
    0.20,  # Медицина
])

# 4. Расчёт интегрального индекса
df.fillna(0, inplace=True)
df['Индекс_привлекательности'] = scaled_features.dot(beta)
df.fillna(0, inplace=True)
df['Индекс_антипривлекательности'] = 1 - df['Индекс_привлекательности']
df.fillna(0, inplace=True)
# 5. Генерация потоков с новым индексом
np.random.seed(0)
synth_fdf = gravity_singly.generate(
    df,
    tile_id_column='name',
    tot_outflows_column='Исходящая', 
    relevance_column='Индекс_антипривлекательности',
    out_format='flows'
)

# Постобработка
# synth_fdf['flow'] = round(synth_fdf['flow'])
# synth_fdf = synth_fdf[synth_fdf['flow'] > 0]

synth_fdf.plot_flows(tiles='CartoDB dark_matter', min_flow=100, flow_weight=0.5, zoom=9)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Создаём копию исходного DataFrame
df = gdf1.copy()

# 1. Рассчитываем нормированные показатели на 1000 человек
population = df['Численность населения (чел.)']

# Социально-культурные фичи

df['Рестораны'] = df['Количество мест в ресторанах кафе барах (место)'] / population
df['Спорт'] = df['Число спортивных сооружений (шт.)'] / population
df['Быт'] = df['Объекты бытового обслуживания (шт.)'] / population
df['Инвестиции'] = df['Инвестиции в основной капитал (тыс. руб.)'] / population
df['Медицина'] = df['Лечебно-профилактические организации (шт.)'] / population

# Замена бесконечных значений (если population=0)
df.replace([np.inf, -np.inf], 0, inplace=True)

# 2. Нормализация признаков
features = [
    'Рестораны',
    'Спорт',
    'Быт',
    'Инвестиции',
    'Медицина',
]

scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(df[features])

# 3. Весовые коэффициенты 
beta = np.array([
    0.05,  # Рестораны
    0.05,  # Спорт
    0.05,  # Быт
    0.05,  # Инвестиции
    4.80,  # Медицина
])

# 4. Расчёт интегрального индекса
df.fillna(0, inplace=True)
df['Индекс_привлекательности'] = scaled_features.dot(beta)
df.fillna(0, inplace=True)
df['Индекс_антипривлекательности'] = 1 - df['Индекс_привлекательности']
df.fillna(0, inplace=True)
# 5. Генерация потоков с новым индексом
np.random.seed(0)
synth_fdf2 = gravity_singly.generate(
    df,
    tile_id_column='name',
    tot_outflows_column='Исходящая', 
    relevance_column='Индекс_антипривлекательности',
    out_format='flows'
)

# Постобработка
# synth_fdf2['flow'] = round(synth_fdf2['flow'])
# synth_fdf2 = synth_fdf2[synth_fdf2['flow'] > 0]

synth_fdf2.plot_flows(tiles='CartoDB dark_matter', min_flow=100, flow_weight=0.5, zoom=9)

In [ ]:
# Создаем множество кортежей для быстрого поиска
common_keys = set(zip(synth_fdf2['origin'], synth_fdf2['destination']))

# Фильтруем synth_fdf2, оставляя только строки, где комбинация origin и destination есть в synth_fdf
filtered_synth_fdf = synth_fdf[
    synth_fdf.apply(lambda row: (row['origin'], row['destination']) in common_keys, axis=1)
]

# Результат
filtered_synth_fdf['flow'] = filtered_synth_fdf['flow'] - synth_fdf2['flow']
filtered_synth_fdf.fillna(0, inplace=True)
filtered_synth_fdf.plot_flows(tiles='CartoDB dark_matter', flow_weight=0.5, zoom=9)

In [ ]:
df[['name', 'Индекс_привлекательности', 'Индекс_антипривлекательности']].sort_values('Индекс_привлекательности')

### Statistics

In [ ]:
gdf1[gdf1['Исходящая'] > 0]['Исходящая'].hist()

In [ ]:
synth_fdf.groupby('origin')['flow'].sum().hist()